# Week 04 — Neural Network Surrogates

Train NN surrogates for each of the 8 functions. One architecture family (2-hidden-layer MLP, Tanh), 4 regularization variants, 2 widths.

For each function:
1. 5-fold CV across `H ∈ {16, 32}` × variants `{plain, dropout, wd, ensemble}`
2. Pick best (H, variant) by CV RMSE
3. Compare against baseline `Y.std()` — only save if it beats baseline
4. Refit on all data, save to `../models/function_N_nn.pt`
5. Save per-dim gradient at current best point (for reflection Q5)

**F1 extra:** train sign classifier (positive vs zero-y) for Q3/Q6.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
sys.path.append('../src')
import nn_models as nm

MODELS_DIR = Path('../models/week_04')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

WIDTHS = [16, 32]
VARIANTS = list(nm.VARIANTS)

def load(n):
    X = np.load(f'../data/function_{n}/initial_inputs.npy')
    Y = np.load(f'../data/function_{n}/initial_outputs.npy')
    return X, Y

In [2]:
def search_and_save(n, verbose=True):
    X, Y = load(n)
    baseline = float(Y.std())
    results = []
    for H in WIDTHS:
        for v in VARIANTS:
            rmse = nm.cv_rmse(X, Y, v, hidden=H, n_splits=5, seed=0)
            results.append((rmse, H, v))
    results.sort(key=lambda r: r[0])
    best_rmse, best_H, best_v = results[0]
    improvement = (baseline - best_rmse) / baseline * 100
    beat_baseline = best_rmse < baseline

    if verbose:
        print(f'F{n}: {X.shape[0]} pts, {X.shape[1]}D, baseline RMSE = {baseline:.4f}')
        print(f'  All configs (sorted):')
        for r, H, v in results:
            mark = ' ← BEST' if (r, H, v) == (best_rmse, best_H, best_v) else ''
            print(f'    H={H:3d}  {v:10s}  CV RMSE = {r:.4f}{mark}')
        status = '✓ beats baseline' if beat_baseline else '✗ WORSE than baseline'
        print(f'  → best: H={best_H}, {best_v}, RMSE={best_rmse:.4f} ({improvement:+.1f}% vs baseline) {status}')

    # Fit final model on all data
    models, meta = nm.fit_final(X, Y, best_v, best_H)
    meta['cv_rmse'] = best_rmse
    meta['baseline_rmse'] = baseline
    meta['beats_baseline'] = beat_baseline
    meta['all_configs'] = [{'hidden': H, 'variant': v, 'rmse': r} for r, H, v in results]

    # Gradient at current best point
    x_best = X[Y.argmax()]
    grad = nm.gradient_at(models, meta, x_best)
    meta['gradient_at_best'] = grad.tolist()
    meta['x_best'] = x_best.tolist()
    meta['y_best'] = float(Y.max())

    if verbose:
        print(f'  Gradient dY/dx at best point: {np.round(grad, 3).tolist()}')

    nm.save(models, meta, MODELS_DIR / f'function_{n}_nn.pt')
    return meta

## Train all 8 functions

In [3]:
all_meta = {}
for n in range(1, 9):
    print('=' * 72)
    all_meta[n] = search_and_save(n, verbose=True)
    print()

F1: 13 pts, 2D, baseline RMSE = 0.0019
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.0028 ← BEST
    H= 32  dropout     CV RMSE = 0.0029
    H= 32  ensemble    CV RMSE = 0.0047
    H= 16  ensemble    CV RMSE = 0.0048
    H= 16  wd          CV RMSE = 0.0052
    H= 32  wd          CV RMSE = 0.0053
    H= 32  plain       CV RMSE = 0.0054
    H= 16  plain       CV RMSE = 0.0054
  → best: H=16, dropout, RMSE=0.0028 (-46.0% vs baseline) ✗ WORSE than baseline
  Gradient dY/dx at best point: [0.009999999776482582, 0.004999999888241291]



F2: 13 pts, 2D, baseline RMSE = 0.2324
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.2102 ← BEST
    H= 16  ensemble    CV RMSE = 0.2132
    H= 16  dropout     CV RMSE = 0.2157
    H= 32  dropout     CV RMSE = 0.2249
    H= 16  wd          CV RMSE = 0.2254
    H= 32  wd          CV RMSE = 0.2348
    H= 16  plain       CV RMSE = 0.2357
    H= 32  plain       CV RMSE = 0.2386
  → best: H=32, ensemble, RMSE=0.2102 (+9.6% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.36800000071525574, 2.313999891281128]



F3: 18 pts, 3D, baseline RMSE = 0.0784
  All configs (sorted):
    H= 32  ensemble    CV RMSE = 0.0845 ← BEST
    H= 32  dropout     CV RMSE = 0.0853
    H= 32  wd          CV RMSE = 0.0875
    H= 16  dropout     CV RMSE = 0.0921
    H= 32  plain       CV RMSE = 0.0929
    H= 16  ensemble    CV RMSE = 0.0966
    H= 16  plain       CV RMSE = 0.1108
    H= 16  wd          CV RMSE = 0.1167
  → best: H=32, ensemble, RMSE=0.0845 (-7.7% vs baseline) ✗ WORSE than baseline


  Gradient dY/dx at best point: [-0.32100000977516174, 1.1089999675750732, 0.628000020980835]



...

  Gradient dY/dx at best point: [0.24699999392032623, -0.05299999937415123, -1.031000018119812, -1.3040000200271606, -0.7310000061988831]



F7: 33 pts, 6D, baseline RMSE = 0.4029
  All configs (sorted):
    H= 32  dropout     CV RMSE = 0.3497 ← BEST
    H= 32  ensemble    CV RMSE = 0.3681
    H= 16  ensemble    CV RMSE = 0.3837
    H= 32  wd          CV RMSE = 0.3842
    H= 32  plain       CV RMSE = 0.3875
    H= 16  dropout     CV RMSE = 0.3902
    H= 16  wd          CV RMSE = 0.4367
    H= 16  plain       CV RMSE = 0.4449
  → best: H=32, dropout, RMSE=0.3497 (+13.2% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.22100000083446503, 0.40799999237060547, 0.07599999755620956, -0.29899999499320984, -0.578000009059906, 0.024000000208616257]



F8: 43 pts, 8D, baseline RMSE = 1.0476
  All configs (sorted):
    H= 16  dropout     CV RMSE = 0.3996 ← BEST
    H= 32  dropout     CV RMSE = 0.4014
    H= 32  ensemble    CV RMSE = 0.4088
    H= 32  plain       CV RMSE = 0.4245
    H= 32  wd          CV RMSE = 0.4267
    H= 16  ensemble    CV RMSE = 0.4306
    H= 16  wd          CV RMSE = 0.4925
    H= 16  plain       CV RMSE = 0.4956
  → best: H=16, dropout, RMSE=0.3996 (+61.9% vs baseline) ✓ beats baseline


  Gradient dY/dx at best point: [-0.4659999907016754, -0.20000000298023224, -0.6679999828338623, -0.14100000262260437, 0.013000000268220901, 0.0689999982714653, -0.39800000190734863, 0.16500000655651093]



## F1 sign classifier

F1 is numerically ~0 for almost all points. Training an NN classifier on sign(y > 0) gives the analyze step a map of "where is the function positive" — useful for Q3/Q6 in the reflection.

In [4]:
X1, Y1 = load(1)
pos_frac = (Y1 > 0).mean()
print(f'F1: {len(Y1)} pts, {(Y1 > 0).sum()} positive, {(Y1 <= 0).sum()} zero-or-negative ({pos_frac:.0%} positive)')

if 0 < pos_frac < 1:
    clf, loo_acc = nm.train_sign_classifier(X1, Y1, hidden=16)
    nm.save_classifier(clf, loo_acc, d_in=X1.shape[1], hidden=16, path=MODELS_DIR / 'function_1_classifier.pt')
    print(f'Sign classifier trained. LOO accuracy = {loo_acc:.2%}')
else:
    print('Classifier skipped (all samples are one class).')

F1: 13 pts, 9 positive, 4 zero-or-negative (69% positive)


Sign classifier trained. LOO accuracy = 61.54%


## Summary

In [5]:
print(f"{'F':>2}  {'H':>3}  {'variant':10s}  {'CV RMSE':>10s}  {'baseline':>10s}  {'improve%':>8s}  {'beats?':>6s}")
print('-' * 62)
for n, m in all_meta.items():
    improve = (m['baseline_rmse'] - m['cv_rmse']) / m['baseline_rmse'] * 100
    mark = '✓' if m['beats_baseline'] else '✗'
    print(f'{n:>2}  {m["hidden"]:>3}  {m["variant"]:10s}  {m["cv_rmse"]:>10.4f}  {m["baseline_rmse"]:>10.4f}  {improve:>+7.1f}%  {mark:>6s}')

 F    H  variant        CV RMSE    baseline  improve%  beats?
--------------------------------------------------------------
 1   16  dropout         0.0028      0.0019    -46.0%       ✗
 2   32  ensemble        0.2102      0.2324     +9.6%       ✓
 3   32  ensemble        0.0845      0.0784     -7.7%       ✗
 4   16  ensemble        4.2366      8.2702    +48.8%       ✓
 5   32  plain         106.2675    425.9412    +75.1%       ✓
 6   16  ensemble        0.3463      0.5488    +36.9%       ✓
 7   32  dropout         0.3497      0.4029    +13.2%       ✓
 8   16  dropout         0.3996      1.0476    +61.9%       ✓
